# SEIS 766: Vision AI (SP26)
## Final Project: Latent Space Traversal
Dante Razo, razo3843@stthomas.edu

## Configuring Environment for GPU Acceleration

In [11]:
from torch import device as torch_device
from torch import cuda

# select device and verify CUDA visibility
device: torch_device = torch_device(device="cuda" if cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("CUDA not detected!")

# log hardware info
print(
    f"Using {str(object=device).upper()} on {cuda.get_device_name(device=cuda.current_device())}"
)

Using CUDA on NVIDIA GeForce RTX 5090


In [12]:
from os import environ

# configure keras
environ["KERAS_BACKEND"] = "torch"
environ["KERAS_TORCH_DEVICE"] = "cuda"

# reduce verbosity
enable_cuda_debug_blocking: bool = False
if enable_cuda_debug_blocking:
    environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [13]:
# check driver and GPU status
!uname -a && echo
!nvidia-smi

Linux ovedur 6.6.87.2-microsoft-standard-WSL2 #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025 x86_64 x86_64 x86_64 GNU/Linux

Sat Apr 25 16:13:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.04              Driver Version: 596.21         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:0A:00.0  On |                  N/A |
|  0%   50C    P0             85W /  460W |    4398MiB /  32607MiB |      6%      Default |
|         

In [14]:
from keras import backend

# configure keras for torch support
backend.set_image_data_format(data_format="channels_last")
backend.set_floatx(value="float32")
backend.clear_session()

In [15]:
from keras.mixed_precision import Policy, set_global_policy

# set global precision policy for torch
policy: Policy = Policy(name="float32")
set_global_policy(policy=policy)

# verify global data types / policies
print(f"Compute Data Type: {policy.compute_dtype}")
print(f"Variable Data Type: {policy.variable_dtype}")

Compute Data Type: float32
Variable Data Type: float32


In [16]:
from keras import config as keras_config

# configure keras
print(f"Keras Backend: {str(object=keras_config.backend()).capitalize()}")

Keras Backend: Torch


## Loading Convolutional Neural Networks

In [17]:
from keras.models import Sequential, load_model

# load model into memory
cnn1: Sequential = load_model(
    filepath="CNN/melanoma_cnn_arch1_35epochs.keras", compile=False
)  # type: ignore

# modify model object's attribute
cnn1.name = "cnn_shallow"

# preview architecture
cnn1.summary()

Model: "cnn_shallow"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,841 (108.75 KB)

 Trainable params: 27,841 (108.75 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
from keras.models import Sequential, load_model

# load model into memory
cnn2: Sequential = load_model(
    filepath="CNN/melanoma_cnn_arch2_35epochs.keras", compile=False
)  # type: ignore

# modify model object's attribute
cnn2.name = "cnn_deep"

# preview architecture
cnn2.summary()

Model: "cnn_deep"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,889 (429.25 KB)

 Trainable params: 109,889 (429.25 KB)

 Non-trainable params: 0 (0.00 B)

## Loading Data

In [19]:
from pandas import DataFrame, read_csv

# load ground truth
labels_df: DataFrame = read_csv(
    filepath_or_buffer="../data/dante/train-ground-truth/ISIC_2020_Training_GroundTruth_v2.csv",
    usecols=["image_name", "target"],
)

# derive new columns
labels_df["filename"] = labels_df["image_name"] + ".jpg"


# preview df
labels_df

,image_name,target,filename
0,ISIC_2637011,0,ISIC_2637011.jpg
1,ISIC_0015719,0,ISIC_0015719.jpg
2,ISIC_0052212,0,ISIC_0052212.jpg
3,ISIC_0068279,0,ISIC_0068279.jpg
4,ISIC_0074268,0,ISIC_0074268.jpg
...,...,...,...
33121,ISIC_9999134,0,ISIC_9999134.jpg
33122,ISIC_9999320,0,ISIC_9999320.jpg
33123,ISIC_9999515,0,ISIC_9999515.jpg
33124,ISIC_9999666,0,ISIC_9999666.jpg


In [20]:
from tarfile import open as untar
from os.path import basename

# define jpg tarball location
real_jpg_tar: str = "../data/dante/train/jpg.tar"


# keep only real melanoma rows that exist in the JPG tar
with untar(name=real_jpg_tar, mode="r") as train_tar:
    tar_filenames: set[str] = {
        basename(p=member.name)
        for member in train_tar.getmembers()
        if member.isfile() and member.name.lower().endswith(".jpg")
    }

# select only ground truth images are present in the tarball
real_df: DataFrame = labels_df.loc[
    (labels_df["target"].astype(dtype=int) == 1)
    & (labels_df["filename"].isin(values=tar_filenames))
]

# retain original csv row index for future reference
# real_df["csv_row"] = real_df.index

# compare size of real_df with tarball contents
print(
    f"Samples reduced by {1 - len(real_df) / len(tar_filenames):.2%} "
    f"from {len(tar_filenames)} to {len(real_df)} after filtering!"
)

# preview data
real_df

Samples reduced by 98.24% from 33126 to 584 after filtering!


,image_name,target,filename
91,ISIC_0149568,1,ISIC_0149568.jpg
235,ISIC_0188432,1,ISIC_0188432.jpg
314,ISIC_0207268,1,ISIC_0207268.jpg
399,ISIC_0232101,1,ISIC_0232101.jpg
459,ISIC_0247330,1,ISIC_0247330.jpg
...,...,...,...
32969,ISIC_9955163,1,ISIC_9955163.jpg
33000,ISIC_9963177,1,ISIC_9963177.jpg
33014,ISIC_9967383,1,ISIC_9967383.jpg
33050,ISIC_9978107,1,ISIC_9978107.jpg


## Preparing Data for GPU-Accelerated Inference

In [21]:
from PIL import Image
from numpy import asarray, float32, stack, ndarray

# derive image dimensions from model input shape
img_height: int = cnn1.input_shape[1]
img_width: int = cnn1.input_shape[2]


# define preprocessing function
def load_image_batch(paths: list[str]) -> ndarray:
    # init empty list
    batch: list[ndarray] = []
    for path in paths:
        # load image into memory
        img: Image.Image = (
            Image.open(fp=path).convert("RGB").resize(size=(img_width, img_height))
        )

        # normalize pixel values
        x: ndarray = asarray(img, dtype=float32) / 255.0

        # append to batch list
        batch.append(x)

    # return image batch
    return stack(arrays=batch, axis=0)

In [22]:
from keras import Model

cnn1_encoder = Model(
    inputs=cnn1.input, outputs=cnn1.layers[-3].output, name="cnn1_encoder"
)
cnn2_encoder = Model(
    inputs=cnn2.input, outputs=cnn2.layers[-3].output, name="cnn2_encoder"
)

AttributeError: 'Sequential' object has no attribute 'input'

## Initial Exploration of Latent Space